# 07 - YOLO Plate Segmentation Training

Notebook ini membangun dataset polygon-only, melatih YOLO11n segmentation, mengevaluasi checkpoint, dan memvisualisasikan perspective crop untuk ANPR. Logic reusable tetap berada di `ml/training/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'ml').is_dir())
SOURCE_ROOT = PROJECT_ROOT / 'TEMPORARY/Test V2'
DATASET_ROOT = PROJECT_ROOT / 'data/processed/plate_segmentation_v1'
DATASET_YAML = DATASET_ROOT / 'data.yaml'
RUN_ROOT = PROJECT_ROOT / 'ml/experiments/yolo_plate_segmentation/yolo11n_plate_seg_v1'
BEST_MODEL = RUN_ROOT / 'weights/best.pt'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Project:', PROJECT_ROOT)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Build polygon-only dataset

In [ ]:
builder = PROJECT_ROOT / 'ml/training/build_plate_segmentation_dataset.py'
subprocess.run([sys.executable, str(builder)], cwd=PROJECT_ROOT, check=True)

In [ ]:
audit = []
for split in ('train', 'valid', 'test'):
    images = list((DATASET_ROOT / split / 'images').glob('*'))
    labels = list((DATASET_ROOT / split / 'labels').glob('*.txt'))
    polygons = sum(len(path.read_text(encoding='utf-8').splitlines()) for path in labels)
    audit.append({'split': split, 'images': len(images), 'label_files': len(labels), 'polygons': polygons})
pd.DataFrame(audit)

## 2. Inspect segmentation labels

In [ ]:
def draw_label(image_path: Path, label_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    for line in label_path.read_text(encoding='utf-8').splitlines():
        values = [float(value) for value in line.split()[1:]]
        polygon = np.array(values, dtype=np.float32).reshape(-1, 2)
        polygon[:, 0] *= width
        polygon[:, 1] *= height
        cv2.polylines(image, [polygon.astype(np.int32)], True, (0, 255, 0), 3)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

label_paths = sorted((DATASET_ROOT / 'train/labels').glob('*.txt'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, label_path in zip(axes.flat, label_paths):
    image_path = next((DATASET_ROOT / 'train/images').glob(f'{label_path.stem}.*'))
    axis.imshow(draw_label(image_path, label_path))
    axis.set_title(image_path.name[:32])
    axis.axis('off')
plt.tight_layout()

## 3. Train YOLO11n segmentation

Ubah `RUN_TRAINING` menjadi `True` hanya saat ingin memulai atau mengulang training. Gunakan kernel `.venv_yolo`.

In [ ]:
RUN_TRAINING = False
EPOCHS = 15

if RUN_TRAINING:
    trainer = PROJECT_ROOT / 'ml/training/train_plate_segmentation.py'
    subprocess.run(
        [sys.executable, str(trainer), '--epochs', str(EPOCHS), '--device', str(DEVICE)],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Training skipped. Set RUN_TRAINING = True to train.')

## 4. Review training metrics

In [ ]:
results_path = RUN_ROOT / 'results.csv'
if results_path.is_file():
    metrics = pd.read_csv(results_path)
    metrics.columns = metrics.columns.str.strip()
    display(metrics[['epoch', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)', 'metrics/precision(M)', 'metrics/recall(M)']])
    metrics.plot(x='epoch', y=['metrics/mAP50(M)', 'metrics/mAP50-95(M)'], marker='o', figsize=(10, 5), grid=True)
else:
    print('No results.csv yet. Run training first.')

## 5. Test checkpoint and visualize masks

In [ ]:
if not BEST_MODEL.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {BEST_MODEL}')
model = YOLO(str(BEST_MODEL))
sample_paths = sorted((DATASET_ROOT / 'test/images').glob('*'))[:6]
predictions = model.predict(sample_paths, imgsz=960, conf=0.25, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, prediction in zip(axes.flat, predictions):
    axis.imshow(cv2.cvtColor(prediction.plot(), cv2.COLOR_BGR2RGB))
    axis.set_title(Path(prediction.path).name[:32])
    axis.axis('off')
plt.tight_layout()

## 6. Perspective rectification from predicted masks

In [ ]:
def warp_mask(image: np.ndarray, polygon: np.ndarray):
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    rectangle = cv2.boxPoints(cv2.minAreaRect(points)).astype(np.float32)
    sums, differences = rectangle.sum(axis=1), np.diff(rectangle, axis=1).reshape(-1)
    ordered = np.array([rectangle[np.argmin(sums)], rectangle[np.argmin(differences)], rectangle[np.argmax(sums)], rectangle[np.argmax(differences)]], dtype=np.float32)
    tl, tr, br, bl = ordered
    width = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))
    height = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))
    if width < 24 or height < 12:
        return None
    target = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    return cv2.warpPerspective(image, cv2.getPerspectiveTransform(ordered, target), (width, height), borderMode=cv2.BORDER_REPLICATE)

rows = []
for prediction in predictions:
    image = cv2.imread(str(prediction.path))
    if prediction.masks is None:
        continue
    for polygon in prediction.masks.xy:
        rectified = warp_mask(image, polygon)
        if rectified is not None:
            rows.append((image, rectified))

fig, axes = plt.subplots(min(4, len(rows)), 2, figsize=(12, 4 * min(4, len(rows))))
for row_axes, (source, crop) in zip(np.atleast_2d(axes), rows[:4]):
    row_axes[0].imshow(cv2.cvtColor(source, cv2.COLOR_BGR2RGB))
    row_axes[0].set_title('Source')
    row_axes[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    row_axes[1].set_title('Segmented + rectified crop')
    for axis in row_axes:
        axis.axis('off')
plt.tight_layout()

## 7. Package for the web application

Setelah checkpoint tervalidasi, jalankan cell berikut. Packager menyalin `best.pt` dan recognizer PaddleOCR ke `ml/weights/anpr/` serta membuat manifest checksum.

In [ ]:
PACKAGE_MODELS = False
if PACKAGE_MODELS:
    packager = PROJECT_ROOT / 'infrastructure/scripts/package_anpr_models.py'
    subprocess.run([sys.executable, str(packager)], cwd=PROJECT_ROOT, check=True)
else:
    print('Packaging skipped. Set PACKAGE_MODELS = True after validating the model.')